# Kernel matrices: similarity at a glance
**Machine Learning · Week 4 (Kernels and support vector machines) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w4d4.html

An SVM never looks at the data itself, only at the **kernel matrix**: the similarity of every pair of points.
Today we build those matrices, read them, and hand them to an SVM directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC

Xc, yc = make_circles(n_samples=200, noise=0.08, factor=0.45, random_state=1)
order = np.lexsort((np.arctan2(Xc[:, 1], Xc[:, 0]), yc))     # by class, then by angle around the circle
X, y = Xc[order], yc[order]
K = rbf_kernel(X, gamma=2)
print("shape:", K.shape, "  symmetric?", np.allclose(K, K.T), "  diagonal all 1?", np.allclose(np.diag(K), 1))

plt.figure(figsize=(4.6, 3.9))
plt.imshow(K, cmap="Purples")
plt.colorbar(label="similarity k(x, x')")
plt.title("Kernel matrix, points sorted by class")
plt.grid(False)
plt.show()

## Too wide, just right, too narrow
The off-diagonal average says how similar *different* points look. Near 1: everything looks alike.
Near 0: everything looks unique. Both extremes carry no useful information.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3), layout="constrained")
for ax, gamma in zip(axes, [0.001, 2, 1000]):
    Kg = rbf_kernel(X, gamma=gamma)
    off = Kg[~np.eye(len(Kg), dtype=bool)].mean()
    ax.imshow(Kg, cmap="Purples", vmin=0, vmax=1)
    ax.set_title(f"gamma = {gamma}\noff-diagonal mean {off:.3f}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## What makes a valid kernel?
A real kernel is an inner product, so its matrix is **positive semidefinite**: no negative eigenvalues.
A made-up "similarity" like minus the distance fails this test.

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

eig_rbf = np.linalg.eigvalsh(rbf_kernel(X, gamma=2))
eig_bad = np.linalg.eigvalsh(-euclidean_distances(X))
print(f"RBF kernel:          smallest eigenvalue {eig_rbf.min():+.2e}  (zero up to rounding)")
print(f"minus the distance:  smallest eigenvalue {eig_bad.min():+.2f}  (negative: not a kernel)")

## Hand SVC the matrix
`kernel="precomputed"`: fit with the train × train matrix; predict with a **test × train** matrix
(each test point's similarity to every training point). This is exactly the recipe for quantum kernels.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(Xc, yc, test_size=0.3, random_state=0)
K_train = rbf_kernel(X_tr, X_tr, gamma=2)          # 140 x 140
K_test = rbf_kernel(X_te, X_tr, gamma=2)           # 60 x 140
model = SVC(kernel="precomputed").fit(K_train, y_tr)
print("K_train", K_train.shape, " K_test", K_test.shape)
print("test accuracy with a precomputed kernel:", round(model.score(K_test, y_te), 3))

## Your turn: build the matrix yourself
Write `rbf_matrix(A, B, gamma)` with NumPy broadcasting: entry $(i, j)$ is $e^{-\gamma\|a_i - b_j\|^2}$.
Check it against scikit-learn, then use it for a precomputed SVM.

In [ ]:
def rbf_matrix(A, B, gamma):
    return np.zeros((len(A), len(B)))   # TODO: squared distances between every row of A and every row of B, then exp

print("matches scikit-learn?", np.allclose(rbf_matrix(X_tr, X_tr, 2), K_train))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def rbf_matrix(A, B, gamma):
    sq = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)    # shape (len(A), len(B))
    return np.exp(-gamma * sq)

print("matches scikit-learn?", np.allclose(rbf_matrix(X_tr, X_tr, 2), K_train))
mine = SVC(kernel="precomputed").fit(rbf_matrix(X_tr, X_tr, 2), y_tr)
print("test accuracy:", round(mine.score(rbf_matrix(X_te, X_tr, 2), y_te), 3))